# 02d — Mejorar la A3 con negativos difíciles

El diagnóstico de errores de la A3 de Holes (`reports/cisutac/errores/README.md`, issue #5) encontró que el mAP@0.5 bajo **no se debe a que el modelo no vea los agujeros**: entre sus 100 predicciones más confiables por foto llega al 87 %. El problema es que la confianza no separa los aciertos de los errores, sobre todo de las **alucinaciones** sobre texturas que parecen agujeros (ojalillos, lunares, ranuras, pliegues). Con YOLO26n, que subió la A3 de Holes de 0,131 a 0,226 en valid, las alucinaciones siguen siendo el error que más mAP cuesta (Δ +0,158).

Además, la ablación del 02 mostró que sumar más tiles sanos **sorteados al azar** (neg 3) no reduce las alucinaciones. La hipótesis de este notebook es que importa **cuáles** tiles sanos ve el modelo, no cuántos:

| ID | Tiles sanos de entrenamiento | Qué responde |
|---|---|---|
| **A3** | 1 por tile con defecto, **sorteados** (igual que el 02) | Referencia |
| **M2-100** | La misma cantidad, todos **difíciles**: los tiles sanos de train donde la A3 más se dispara | ¿Enseñarle justo lo que confunde reduce las alucinaciones? |
| **M2-50** | La misma cantidad, mitad difíciles y mitad sorteados | Opción conservadora: sigue viendo tela sana común y diluye posibles agujeros sin anotar |

**Reglas:** YOLO26n, semilla 0, el mismo presupuesto de imágenes vistas que la A3 del 02 y la misma evaluación por foto (SAHI, coordenadas de la original). El checkpoint es el `best.pt` de Ultralytics en las tres corridas, para que lo único que cambie sean los negativos; la selección de checkpoint por mAP por foto es otra mejora (`scripts/select_checkpoint.py`, issue #5) y se puede aplicar después a la ganadora. **Test sigue cerrado**: la variante que gane se evalúa después en test y con tres semillas, como en el 02c.

> **Negativo difícil** (*hard negative mining*): en vez de mostrarle al modelo ejemplos sanos cualquiera, se le muestran los sanos que hoy confunde con un defecto. Es una idea clásica de detección (Viola-Jones; Felzenszwalb et al.) y del entrenamiento de SSD.

## 0. Parámetros

In [ ]:
DATASET = "holes"        # "holes" o "spots"
MODO_PRUEBA = True       # True: 1 época y pocas imágenes, sólo para verificar el pipeline

MODELO = "yolo26n.pt"    # mismo detector que la corrida YOLO26 del 02
SEMILLA = 0
EPOCAS_BASE, PACIENCIA_BASE = 100, 30   # presupuesto de la foto completa; los tiles se igualan en imágenes vistas
TILE, SOLAPAMIENTO, MIN_VISIBLE = 640, 0.2, 0.5
NEG_POR_POS = 1.0                        # misma cantidad de negativos que la A3 de Holes
BATCH = 16
CONF_EVAL = 0.001

FRACCIONES_DIFICILES = [1.0, 0.5]        # fracción de los negativos que son difíciles; el resto se sortea
CONF_MINERIA = 0.01                      # confianza mínima para puntuar un tile sano

if MODO_PRUEBA:
    EPOCAS_BASE, PACIENCIA_BASE = 1, 1
    LIMITE_EVAL, MAX_IMAGENES_TILES = 3, 6
else:
    LIMITE_EVAL, MAX_IMAGENES_TILES = None, None
SUFIJO = "_prueba" if MODO_PRUEBA else ""

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
import cisutac_baseline as cb
import analyze_detection_errors as ade

pd.set_option("display.max_columns", 30)
print("Datos:", cb.DATA)
print("Resultados:", cb.RESULTS)
print("Dispositivo:", cb.pick_device())

if not (cb.derived_dir(DATASET) / "data.yaml").exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "prepare_cisutac_yolo.py")], check=True)
n_completas = MAX_IMAGENES_TILES or len(cb.load_ground_truth(DATASET, "train")["images"])
SALIDA = cb.RESULTS / "mejora_a3" / f"{DATASET}{SUFIJO}"
SALIDA.mkdir(parents=True, exist_ok=True)


def nombre(variante):
    # Igual que el 02: con otro modelo que YOLO11n, el nombre lo incluye.
    modelo = "" if MODELO == "yolo11n.pt" else f"_{Path(MODELO).stem}"
    return f"{DATASET}_{variante}{modelo}{SUFIJO}"


def presupuesto(stats):
    # Igual que el 02: épocas y paciencia equivalentes en imágenes vistas a EPOCAS_BASE sobre fotos completas.
    epocas = cb.equivalent_epochs(EPOCAS_BASE, n_completas, stats["train"]["tiles_totales"],
                                  minimum=1 if MODO_PRUEBA else 10)
    return epocas, max(1, round(PACIENCIA_BASE * epocas / EPOCAS_BASE))


def entrenar(variante, data_yaml, stats):
    epocas, paciencia = presupuesto(stats)
    return cb.train(nombre(variante), data_yaml, imgsz=TILE, epochs=epocas, patience=paciencia, batch=BATCH,
                    seed=SEMILLA, model=MODELO)


def evaluar(run_dir):
    # Evaluación por foto en valid con tiles (SAHI), la misma de todo el bloque A.
    return cb.evaluate(f"{Path(run_dir).name}__valid_tiles{TILE}_o{round(100 * SOLAPAMIENTO)}", DATASET,
                       Path(run_dir) / "weights" / "best.pt", split="valid", mode="tiles", tile=TILE,
                       overlap=SOLAPAMIENTO, conf=CONF_EVAL, limit=LIMITE_EVAL)

## 1. A3 de referencia

Tiles de 640 con 20 % de solapamiento y un tile sano sorteado por cada tile con defecto. El nombre es el mismo que en la corrida YOLO26 del 02: si se agrega su Output como *Input*, se reutiliza; si no, se reentrena con la misma semilla y debería dar un mAP@0.5 por foto parecido (0,226 en Holes).

In [ ]:
yaml_azar, stats_azar = cb.build_tile_dataset(DATASET, tile=TILE, overlap=SOLAPAMIENTO, neg_per_pos=NEG_POR_POS,
                                              seed=SEMILLA, min_visible=MIN_VISIBLE, max_images=MAX_IMAGENES_TILES)
display(pd.DataFrame(stats_azar).T)
print("Épocas y paciencia para tiles:", presupuesto(stats_azar))

run_base = entrenar(f"A3_tiles{TILE}_neg{NEG_POR_POS:g}", yaml_azar, stats_azar)
eval_base = evaluar(run_base)
display(cb.metrics_table([eval_base]))

## 2. Minería de negativos difíciles

El `best.pt` de la A3 se corre sobre **todos** los tiles sanos de train (unos 8.900 en Holes), a la misma escala con la que se entrenó. Cada tile recibe como puntaje la confianza máxima que le da el modelo: los de mayor puntaje son los que hoy confunde con un defecto.

Algunos "negativos difíciles" podrían ser **agujeros sin anotar**; enseñarle al modelo que no son defectos lo empeoraría. Por eso se muestran los más difíciles para revisarlos a ojo (no se corrige ninguna etiqueta: sólo se informa) y se prueba también la mezcla mitad y mitad.

In [ ]:
puntajes = cb.score_negative_tiles(DATASET, run_base / "weights" / "best.pt", tile=TILE, overlap=SOLAPAMIENTO,
                                   min_visible=MIN_VISIBLE, max_images=MAX_IMAGENES_TILES, conf=CONF_MINERIA)
serie = pd.Series(puntajes).sort_values(ascending=False)
n_negativos = stats_azar["train"]["tiles_sin_defecto_usados"]
corte = serie.iloc[min(n_negativos, len(serie)) - 1]
print(f"Tiles sanos puntuados: {len(serie)} · con alguna predicción ≥ {CONF_MINERIA}: {(serie > 0).sum()}")
print(f"Negativos por corrida: {n_negativos} · puntaje del más fácil de los {n_negativos} más difíciles: {corte:.3f}")
pd.DataFrame({"archivo": [k[0] for k in serie.index], "x0": [k[1] for k in serie.index],
              "y0": [k[2] for k in serie.index], "puntaje": serie.values}).to_csv(
    SALIDA / "puntajes_tiles_sanos.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(serie.values, bins=50)
ax.axvline(corte, color="C1", ls="--", label="corte de los difíciles")
ax.set_yscale("log")
ax.set_xlabel("confianza máxima de la A3 en el tile sano")
ax.legend()
plt.tight_layout()
plt.savefig(SALIDA / "histograma_puntajes.png", dpi=150)

In [ ]:
from PIL import Image

n = 12
fig, axes = plt.subplots(2, n // 2, figsize=(2.2 * n // 2, 4.8))
for ax, (archivo, x0, y0) in zip(axes.ravel(), serie.index[:n]):
    with Image.open(cb.original_dir(DATASET) / archivo) as im:
        ax.imshow(im.convert("RGB").crop((x0, y0, x0 + TILE, y0 + TILE)))
    ax.set_title(f"{puntajes[(archivo, x0, y0)]:.2f}", fontsize=8)
    ax.axis("off")
fig.suptitle("Tiles sanos de train donde la A3 más se dispara")
plt.tight_layout()
plt.savefig(SALIDA / "tiles_dificiles.png", dpi=120)

## 3. Entrenar con negativos difíciles

Mismos tiles con defecto y **misma cantidad** de tiles sanos que la A3; sólo cambia cuáles. Valid no cambia. Receta, presupuesto y evaluación iguales a la A3.

In [ ]:
evals = {"A3 (sorteados)": eval_base}
for fraccion in FRACCIONES_DIFICILES:
    yaml_d, stats_d = cb.build_tile_dataset(DATASET, tile=TILE, overlap=SOLAPAMIENTO, neg_per_pos=NEG_POR_POS,
                                            seed=SEMILLA, min_visible=MIN_VISIBLE, max_images=MAX_IMAGENES_TILES,
                                            hard_scores=puntajes, hard_fraction=fraccion)
    assert stats_d["train"]["tiles_totales"] == stats_azar["train"]["tiles_totales"], "cambió el tamaño del dataset"
    run = entrenar(f"A3_tiles{TILE}_neg{NEG_POR_POS:g}_dificiles{fraccion:g}", yaml_d, stats_d)
    evals[f"M2-{fraccion:.0%} difíciles"] = evaluar(run)

tabla = cb.metrics_table(list(evals.values()))
tabla.index = list(evals)
tabla.to_csv(SALIDA / "metricas_valid.csv")
display(tabla)
display(cb.size_table(list(evals.values())))

**¿Bajaron las alucinaciones?** El mismo análisis de errores del issue #5 (TIDE), con el umbral de confianza de valid de cada corrida. Si los negativos difíciles funcionan, el Δ mAP de alucinaciones tiene que bajar respecto de la A3 sin que suban mucho los no detectados.

In [ ]:
corridas = [ade.load_run(cb.RESULTS / "evaluaciones" / m["corrida"]) for m in evals.values()]
analisis = [a for a in (ade.analyze(r) for r in corridas if r) if a]
filas_pred, filas_def, filas_delta = ade.table_rows(analisis)
display(pd.DataFrame(filas_pred).set_index("corrida")[["umbral", "predicciones", "pct_aciertos", "pct_localizacion",
                                                       "pct_duplicados", "pct_alucinaciones"]].round(3))
tabla_delta = pd.DataFrame(filas_delta).set_index("corrida").drop(columns="dataset").round(3)
tabla_delta.to_csv(SALIDA / "delta_map.csv")
display(tabla_delta)
ade.plot_summary(analisis, SALIDA / "errores.png")

## 4. Conclusiones

> Completar **después** de correr el notebook sin `MODO_PRUEBA`, con los números de las tablas.

- ¿Los negativos difíciles suben el mAP@0.5 y el AP_small por foto respecto de la A3? ¿Fracción 100 % o 50 %?
- ¿Bajó el Δ mAP de alucinaciones? ¿Subieron los no detectados?
- **Tiles difíciles:** ¿son texturas (ojalillos, lunares, costuras) o parecen agujeros sin anotar?
- **Siguiente paso:** congelar la variante ganadora, aplicarle la selección de checkpoint de `scripts/select_checkpoint.py` si corresponde, y evaluarla en test con tres semillas. La diferencia con la A3 sólo es convincente si supera la dispersión entre semillas que mide el 02c.